In [1]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

2.11.0+cu128 True NVIDIA GeForce RTX 5050 Laptop GPU


In [2]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import random

def load_pairs(path):
    pairs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            num, text = line.split("\t")
            pairs.append((num, text))
    return pairs

num_en = load_pairs("num2txt.txt")
num_ko = load_pairs("num2kor.txt")

print(len(num_en), len(num_ko))
for p in random.sample(num_en, 3):
    print(p)
for p in random.sample(num_ko, 3):
    print(p)

def tok_num(s):
    return list(s)

def tok_en(s):
    return s.split()

def tok_ko(s):
    return list(s)

print(tok_num("12345"))
print(tok_en("twelve thousand three hundred forty five"))
print(tok_ko("만 이천삼백사십오"))

en_vocab = sorted({w for _, t in num_en for w in tok_en(t)})
ko_vocab = sorted({c for _, t in num_ko for c in tok_ko(t)})
print(len(en_vocab), en_vocab)
print(len(ko_vocab), ko_vocab)
print(max(len(tok_en(t)) for _, t in num_en), max(len(tok_ko(t)) for _, t in num_ko))

100000 100000
('5555806540', 'five billion five hundred fifty five million eight hundred six thousand five hundred forty')
('450868445', 'four hundred fifty million eight hundred sixty eight thousand four hundred forty five')
('401379', 'four hundred one thousand three hundred seventy nine')
('72489', '칠만 이천사백팔십구')
('432855421', '사억 삼천이백팔십오만 오천사백이십일')
('4812149', '사백팔십일만 이천백사십구')
['1', '2', '3', '4', '5']
['twelve', 'thousand', 'three', 'hundred', 'forty', 'five']
['만', ' ', '이', '천', '삼', '백', '사', '십', '오']
32 ['billion', 'eight', 'eighteen', 'eighty', 'eleven', 'fifteen', 'fifty', 'five', 'forty', 'four', 'fourteen', 'hundred', 'million', 'nine', 'nineteen', 'ninety', 'one', 'seven', 'seventeen', 'seventy', 'six', 'sixteen', 'sixty', 'ten', 'thirteen', 'thirty', 'thousand', 'three', 'twelve', 'twenty', 'two', 'zero']
16 [' ', '구', '만', '백', '사', '삼', '십', '억', '영', '오', '육', '이', '일', '천', '칠', '팔']
16 21


In [4]:
PAD, SOS, EOS, UNK = 0, 1, 2, 3

class Vocab:
    def __init__(self, tokens, joiner):
        self.itos = ["<pad>", "<sos>", "<eos>", "<unk>"] + sorted(set(tokens))
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.joiner = joiner

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens):
        return [SOS] + [self.stoi.get(t, UNK) for t in tokens] + [EOS]

    def decode(self, ids):
        out = []
        for i in ids:
            if i == EOS:
                break
            if i in (PAD, SOS):
                continue
            out.append(self.itos[i])
        return self.joiner.join(out)

num_vocab = Vocab("0123456789", "")
en_vocab = Vocab([w for _, t in num_en for w in tok_en(t)], " ")
ko_vocab = Vocab([c for _, t in num_ko for c in tok_ko(t)], "")

print(len(num_vocab), len(en_vocab), len(ko_vocab))

ids = en_vocab.encode(tok_en("twelve thousand three hundred forty five"))
print(ids)
print(en_vocab.decode(ids))

ids = ko_vocab.encode(tok_ko("만 이천삼백사십오"))
print(ids)
print(ko_vocab.decode(ids))

ids = num_vocab.encode(tok_num("12345"))
print(ids)
print(num_vocab.decode(ids))

14 36 20
[1, 32, 30, 31, 15, 12, 11, 2]
twelve thousand three hundred forty five
[1, 6, 4, 15, 17, 9, 7, 8, 10, 13, 2]
만 이천삼백사십오
[1, 5, 6, 7, 8, 9, 2]
12345


In [5]:
import random
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

LANGS = {
    "num": (tok_num, num_vocab),
    "en": (tok_en, en_vocab),
    "ko": (tok_ko, ko_vocab),
}

num2en = dict(num_en)
num2ko = dict(num_ko)
shared = sorted(set(num2en) & set(num2ko))

items = {
    ("num", "en"): [(n, n, e) for n, e in num_en],
    ("num", "ko"): [(n, n, k) for n, k in num_ko],
    ("en", "ko"): [(n, num2en[n], num2ko[n]) for n in shared],
}

def split(rows, ratio=(0.8, 0.1, 0.1)):
    rows = rows[:]
    random.shuffle(rows)
    a = int(len(rows) * ratio[0])
    b = a + int(len(rows) * ratio[1])
    return rows[:a], rows[a:b], rows[b:]

def is_rare(n):
    return len(n) <= 3 or n.count("0") >= 4

def oversample(rows, times=10):
    return rows + [r for r in rows if is_rare(r[0])] * (times - 1)

data = {}
for (a, b), rows in items.items():
    tr, va, te = split(rows)
    tr = oversample(tr)
    parts = [[(x, y) for _, x, y in p] for p in (tr, va, te)]
    data[(a, b)] = tuple(parts)
    data[(b, a)] = tuple([(y, x) for x, y in p] for p in parts)

for k, (tr, va, te) in data.items():
    print(k, len(tr), len(va), len(te), tr[0])

class PairDataset(Dataset):
    def __init__(self, pairs, src, tgt):
        s_tok, s_voc = LANGS[src]
        t_tok, t_voc = LANGS[tgt]
        self.data = [
            (torch.tensor(s_voc.encode(s_tok(x))), torch.tensor(t_voc.encode(t_tok(y))))
            for x, y in pairs
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.data[i]

def collate(batch):
    src, tgt = zip(*batch)
    src = pad_sequence(src, batch_first=True, padding_value=PAD)
    tgt = pad_sequence(tgt, batch_first=True, padding_value=PAD)
    return src, tgt

ds = PairDataset(data[("num", "en")][0], "num", "en")
loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=collate)
src, tgt = next(iter(loader))
print(src.shape, tgt.shape)
for s, t in zip(src, tgt):
    print(num_vocab.decode(s.tolist()), "->", en_vocab.decode(t.tolist()))

('num', 'en') 89063 10000 10000 ('962', 'nine hundred sixty two')
('en', 'num') 89063 10000 10000 ('nine hundred sixty two', '962')
('num', 'ko') 89036 10000 10000 ('12', '십이')
('ko', 'num') 89036 10000 10000 ('십이', '12')
('en', 'ko') 17480 1273 1275 ('eight thousand one hundred forty two', '팔천백사십이')
('ko', 'en') 17480 1273 1275 ('팔천백사십이', 'eight thousand one hundred forty two')
torch.Size([4, 11]) torch.Size([4, 15])
53781365 -> fifty three million seven hundred eighty one thousand three hundred sixty five
815264185 -> eight hundred fifteen million two hundred sixty four thousand one hundred eighty five
83613464 -> eighty three million six hundred thirteen thousand four hundred sixty four
12056 -> twelve thousand fifty six


In [6]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, batch_first=True,
                          dropout=dropout if n_layers > 1 else 0)

    def forward(self, src):
        lengths = (src != PAD).sum(1).cpu()
        emb = self.embedding(src)
        packed = pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        out, hidden = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=src.size(1))
        return out, hidden

class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.W = nn.Linear(hid_dim, hid_dim, bias=False)

    def forward(self, query, enc_out, mask):
        scores = torch.bmm(enc_out, self.W(query).unsqueeze(2)).squeeze(2)
        scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), enc_out).squeeze(1)
        return context, weights

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, attention):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, batch_first=True,
                          dropout=dropout if n_layers > 1 else 0)
        self.use_attention = attention
        if attention:
            self.attn = Attention(hid_dim)
            self.combine = nn.Linear(hid_dim * 2, hid_dim)
        self.fc = nn.Linear(hid_dim, vocab_size)

    def forward(self, token, hidden, enc_out, mask):
        emb = self.embedding(token.unsqueeze(1))
        out, hidden = self.rnn(emb, hidden)
        h = out.squeeze(1)
        weights = None
        if self.use_attention:
            context, weights = self.attn(h, enc_out, mask)
            h = torch.tanh(self.combine(torch.cat([h, context], dim=1)))
        return self.fc(h), hidden, weights

class Seq2Seq(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, emb_dim=128, hid_dim=256, n_layers=2, dropout=0.2, attention=True):
        super().__init__()
        self.encoder = Encoder(src_vocab, emb_dim, hid_dim, n_layers, dropout)
        self.decoder = Decoder(tgt_vocab, emb_dim, hid_dim, n_layers, dropout, attention)
        self.tgt_vocab = tgt_vocab

    def forward(self, src, tgt, teacher_forcing=0.5):
        B, T = tgt.shape
        enc_out, hidden = self.encoder(src)
        mask = src != PAD
        outputs = torch.zeros(B, T - 1, self.tgt_vocab, device=src.device)
        token = tgt[:, 0]
        for t in range(1, T):
            logits, hidden, _ = self.decoder(token, hidden, enc_out, mask)
            outputs[:, t - 1] = logits
            if random.random() < teacher_forcing:
                token = tgt[:, t]
            else:
                token = logits.argmax(1)
        return outputs

for att in [False, True]:
    model = Seq2Seq(len(num_vocab), len(en_vocab), attention=att).to(device)
    print(att, sum(p.numel() for p in model.parameters()))

src, tgt = src.to(device), tgt.to(device)
out = model(src, tgt)
print(src.shape, tgt.shape, out.shape)

False 1398052
True 1594916
torch.Size([4, 11]) torch.Size([4, 15]) torch.Size([4, 14, 36])


In [7]:
import time
import copy

def make_loader(src, tgt, part, batch_size=128, shuffle=False):
    idx = {"train": 0, "val": 1, "test": 2}[part]
    ds = PairDataset(data[(src, tgt)][idx], src, tgt)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)

criterion = nn.CrossEntropyLoss(ignore_index=PAD)

def run_epoch(model, loader, optimizer=None, teacher_forcing=0.5):
    train = optimizer is not None
    model.train() if train else model.eval()
    total, count = 0, 0
    with torch.set_grad_enabled(train):
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            out = model(src, tgt, teacher_forcing if train else 0)
            loss = criterion(out.reshape(-1, out.size(-1)), tgt[:, 1:].reshape(-1))
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total += loss.item() * src.size(0)
            count += src.size(0)
    return total / count

def train_model(src, tgt, epochs=10, lr=1e-3, attention=True):
    random.seed(SEED)
    torch.manual_seed(SEED)
    s_voc, t_voc = LANGS[src][1], LANGS[tgt][1]
    model = Seq2Seq(len(s_voc), len(t_voc), attention=attention).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_loader = make_loader(src, tgt, "train", shuffle=True)
    val_loader = make_loader(src, tgt, "val")
    tag = "attn" if attention else "base"
    best_loss, best_state, best_epoch = float("inf"), None, 0
    for epoch in range(1, epochs + 1):
        start = time.time()
        tr = run_epoch(model, train_loader, optimizer)
        va = run_epoch(model, val_loader)
        if va < best_loss:
            best_loss, best_state, best_epoch = va, copy.deepcopy(model.state_dict()), epoch
        print(f"[{src}->{tgt} {tag}] epoch {epoch:2d}  train {tr:.4f}  val {va:.4f}  {time.time() - start:.1f}s")
    model.load_state_dict(best_state)
    print(f"[{src}->{tgt} {tag}] best epoch {best_epoch} (val {best_loss:.4f})")
    return model

In [8]:
@torch.no_grad()
def greedy_decode(model, src, max_len=30):
    model.eval()
    enc_out, hidden = model.encoder(src)
    mask = src != PAD
    B = src.size(0)
    token = torch.full((B,), SOS, dtype=torch.long, device=src.device)
    result = []
    finished = torch.zeros(B, dtype=torch.bool, device=src.device)
    for _ in range(max_len):
        logits, hidden, _ = model.decoder(token, hidden, enc_out, mask)
        token = logits.argmax(1)
        result.append(token)
        finished |= token == EOS
        if finished.all():
            break
    return torch.stack(result, 1)

def translate(model, text, src, tgt):
    s_tok, s_voc = LANGS[src]
    t_voc = LANGS[tgt][1]
    x = torch.tensor([s_voc.encode(s_tok(text))], device=device)
    return t_voc.decode(greedy_decode(model, x)[0].tolist())

def evaluate(model, src, tgt, part="test", show_errors=5):
    t_voc = LANGS[tgt][1]
    s_voc = LANGS[src][1]
    loader = make_loader(src, tgt, part, batch_size=512)
    correct, total, errors = 0, 0, []
    for x, y in loader:
        x = x.to(device)
        pred = greedy_decode(model, x)
        for xi, yi, pi in zip(x.tolist(), y.tolist(), pred.tolist()):
            p, g = t_voc.decode(pi), t_voc.decode(yi)
            if p == g:
                correct += 1
            elif len(errors) < show_errors:
                errors.append((s_voc.decode(xi), g, p))
            total += 1
    acc = correct / total
    print(f"[{src}->{tgt}] {part} accuracy: {acc:.4f} ({correct}/{total})")
    for s, g, p in errors:
        print(f"  {s}\n    answer: {g}\n    pred:   {p}")
    return acc

In [9]:
CONFIG = dict(emb_dim=128, hid_dim=256, n_layers=2, dropout=0.2)
EPOCHS = {("num", "en"): 10, ("en", "num"): 10, ("num", "ko"): 10,
          ("ko", "num"): 10, ("en", "ko"): 30, ("ko", "en"): 30}

final_models, final_attention, table = {}, {}, []
for (src, tgt), ep in EPOCHS.items():
    name = f"{src}2{tgt}"
    cand = {}
    for att in [False, True]:
        m = train_model(src, tgt, epochs=ep, attention=att)
        val_acc = evaluate(m, src, tgt, part="val", show_errors=0)
        test_acc = evaluate(m, src, tgt, part="test", show_errors=3)
        cand[att] = (val_acc, test_acc, m)
    att = max(cand, key=lambda a: cand[a][0])
    final_models[name], final_attention[name] = cand[att][2], att
    table.append((name, cand[False][0], cand[False][1], cand[True][0], cand[True][1], "attn" if att else "base"))

print()
print(f"{'':8s} {'base val':>9s} {'base test':>9s} {'attn val':>9s} {'attn test':>9s}  selected")
for name, bv, bt, av, at, sel in table:
    print(f"{name:8s} {bv:9.4f} {bt:9.4f} {av:9.4f} {at:9.4f}  {sel}")

torch.save({
    "config": CONFIG,
    "attention": final_attention,
    "vocabs": {k: (v.itos, v.joiner) for k, v in [("num", num_vocab), ("en", en_vocab), ("ko", ko_vocab)]},
    "models": {k: m.state_dict() for k, m in final_models.items()},
}, "num2text_models.pt")

[num->en base] epoch  1  train 0.7650  val 0.2481  28.6s
[num->en base] epoch  2  train 0.0616  val 0.0440  26.7s
[num->en base] epoch  3  train 0.0188  val 0.0187  28.7s
[num->en base] epoch  4  train 0.0141  val 0.0147  27.9s
[num->en base] epoch  5  train 0.0108  val 0.0358  28.8s
[num->en base] epoch  6  train 0.0074  val 0.0059  28.3s
[num->en base] epoch  7  train 0.0113  val 0.0048  28.1s
[num->en base] epoch  8  train 0.0021  val 0.0008  29.3s
[num->en base] epoch  9  train 0.0083  val 0.0043  29.4s
[num->en base] epoch 10  train 0.0034  val 0.0140  24.8s
[num->en base] best epoch 8 (val 0.0008)
[num->en] val accuracy: 0.9990 (9990/10000)
[num->en] test accuracy: 0.9988 (9988/10000)
  2000588107
    answer: two billion five hundred eighty eight thousand one hundred seven
    pred:   two billion five hundred eighty eight thousand eight hundred seventeen
  783217786
    answer: seven hundred eighty three million two hundred seventeen thousand seven hundred eighty six
    pred:   

In [10]:
import torch
import torch.nn as nn
from pathlib import Path
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

PAD, SOS, EOS, UNK = 0, 1, 2, 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Vocab:
    def __init__(self, itos, joiner):
        self.itos = itos
        self.stoi = {t: i for i, t in enumerate(itos)}
        self.joiner = joiner

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens):
        return [SOS] + [self.stoi.get(t, UNK) for t in tokens] + [EOS]

    def decode(self, ids):
        out = []
        for i in ids:
            if i == EOS:
                break
            if i in (PAD, SOS):
                continue
            out.append(self.itos[i])
        return self.joiner.join(out)

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, batch_first=True,
                          dropout=dropout if n_layers > 1 else 0)

    def forward(self, src):
        lengths = (src != PAD).sum(1).cpu()
        packed = pack_padded_sequence(self.embedding(src), lengths, batch_first=True, enforce_sorted=False)
        out, hidden = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=src.size(1))
        return out, hidden

class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.W = nn.Linear(hid_dim, hid_dim, bias=False)

    def forward(self, query, enc_out, mask):
        scores = torch.bmm(enc_out, self.W(query).unsqueeze(2)).squeeze(2)
        scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), enc_out).squeeze(1)
        return context, weights

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, attention):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, batch_first=True,
                          dropout=dropout if n_layers > 1 else 0)
        self.use_attention = attention
        if attention:
            self.attn = Attention(hid_dim)
            self.combine = nn.Linear(hid_dim * 2, hid_dim)
        self.fc = nn.Linear(hid_dim, vocab_size)

    def forward(self, token, hidden, enc_out, mask):
        out, hidden = self.rnn(self.embedding(token.unsqueeze(1)), hidden)
        h = out.squeeze(1)
        weights = None
        if self.use_attention:
            context, weights = self.attn(h, enc_out, mask)
            h = torch.tanh(self.combine(torch.cat([h, context], dim=1)))
        return self.fc(h), hidden, weights

class Seq2Seq(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, emb_dim, hid_dim, n_layers, dropout, attention):
        super().__init__()
        self.encoder = Encoder(src_vocab, emb_dim, hid_dim, n_layers, dropout)
        self.decoder = Decoder(tgt_vocab, emb_dim, hid_dim, n_layers, dropout, attention)

TOKENIZERS = {"num": list, "en": str.split, "ko": list}

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "num2text_models.pt").exists():
    PROJECT_DIR = PROJECT_DIR / "project01"

ckpt = torch.load(PROJECT_DIR / "num2text_models.pt", map_location=device)
VOCABS = {k: Vocab(itos, joiner) for k, (itos, joiner) in ckpt["vocabs"].items()}
MODELS = {}
for name, state in ckpt["models"].items():
    src, tgt = name.split("2")
    m = Seq2Seq(len(VOCABS[src]), len(VOCABS[tgt]), **ckpt["config"], attention=ckpt["attention"][name]).to(device)
    m.load_state_dict(state)
    m.eval()
    MODELS[name] = m

@torch.no_grad()
def translate(text, src, tgt, max_len=30):
    model = MODELS[f"{src}2{tgt}"]
    s_voc, t_voc = VOCABS[src], VOCABS[tgt]
    x = torch.tensor([s_voc.encode(TOKENIZERS[src](text))], device=device)
    enc_out, hidden = model.encoder(x)
    mask = x != PAD
    token = torch.tensor([SOS], device=device)
    out = []
    for _ in range(max_len):
        logits, hidden, _ = model.decoder(token, hidden, enc_out, mask)
        token = logits.argmax(1)
        if token.item() == EOS:
            break
        out.append(token.item())
    return t_voc.decode(out)

def num2en(s): return translate(s, "num", "en")
def en2num(s): return translate(s, "en", "num")
def num2ko(s): return translate(s, "num", "ko")
def ko2num(s): return translate(s, "ko", "num")
def en2ko(s): return translate(s, "en", "ko")
def ko2en(s): return translate(s, "ko", "en")

print(num2en("12345"))
print(en2num("twelve thousand three hundred forty five"))
print(num2ko("12345"))
print(ko2num("만 이천삼백사십오"))
print(en2ko("twelve thousand three hundred forty five"))
print(ko2en("만 이천삼백사십오"))

for n in ["0", "7", "19", "100", "1000001", "2026112024", "9876543210"]:
    en, ko = num2en(n), num2ko(n)
    print(f"{n:>11} | {en} | {ko} | {en2num(en)} | {ko2num(ko)}")

twelve thousand three hundred forty five
12345
만 이천삼백사십오
12345
만 이천삼백사십오
twelve thousand three hundred forty five
          0 | zero | 영 | 0 | 0
          7 | seven | 칠 | 7 | 7
         19 | nineteen | 십구 | 19 | 19
        100 | one hundred | 백 | 100 | 100
    1000001 | one million one hundred one | 일 | 1100011 | 1
 2026112024 | two billion twenty six million one hundred twelve thousand twenty four | 이십억 이천육백십일만 이천이십사 | 2026112024 | 2026112024
 9876543210 | nine billion eight hundred seventy six million five hundred forty three thousand two hundred ten | 구십팔억 칠천육백오십사만 삼천이백십 | 9876543210 | 9876543210
